In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("HomeCredit_Serving_Layer") \
    .enableHiveSupport() \
    .getOrCreate()

In [7]:
raw_app = spark.read.parquet("/user/student/home_credit/raw/application_train")
raw_prev = spark.read.parquet("/user/student/home_credit/raw/previous_application")
raw_inst = spark.read.parquet("/user/student/home_credit/raw/installments_payments")
raw_bureau = spark.read.parquet("/user/student/home_credit/raw/bureau")

print("Raw Parquet Data loaded successfully from HDFS!")

Raw Parquet Data loaded successfully from HDFS!


In [20]:
# 1. Application Train
df_app = raw_app.select(
    F.col("SK_ID_CURR").cast("int"),
    F.col("TARGET").cast("int"),
    F.col("NAME_CONTRACT_TYPE").cast("string"),
    F.col("DAYS_BIRTH").cast("int"),
    F.col("OCCUPATION_TYPE").cast("string"),
    F.col("NAME_EDUCATION_TYPE").cast("string"),
    F.col("NAME_FAMILY_STATUS").cast("string"),
    F.col("NAME_HOUSING_TYPE").cast("string"),
    F.col("NAME_INCOME_TYPE").cast("string"),
    F.col("FLAG_OWN_REALTY").cast("string"),
    F.col("AMT_INCOME_TOTAL").cast("double"),
    F.col("AMT_CREDIT").cast("double"),
    F.col("AMT_ANNUITY").cast("double"),
    F.col("AMT_GOODS_PRICE").cast("double"),
    F.col("DAYS_EMPLOYED").cast("int")
)

# 2. Previous Applications
df_prev = raw_prev.select(
    F.col("SK_ID_PREV").cast("int"),
    F.col("SK_ID_CURR").cast("int"),
    F.col("NAME_CONTRACT_STATUS").cast("string"),
    F.col("AMT_CREDIT").cast("double")
)

# 3. Installments Payments
df_inst = raw_inst.select(
    F.col("SK_ID_PREV").cast("int"),
    F.col("SK_ID_CURR").cast("int"),
    F.col("DAYS_INSTALMENT").cast("double"),
    F.col("DAYS_ENTRY_PAYMENT").cast("double"),
    F.col("AMT_INSTALMENT").cast("double"),
    F.col("AMT_PAYMENT").cast("double")
)

# 4. Bureau Data
df_bureau = raw_bureau.select(
    F.col("SK_ID_CURR").cast("int"),
    F.col("CREDIT_ACTIVE").cast("string"),
    F.col("AMT_CREDIT_SUM_DEBT").cast("double"),
    F.col("AMT_CREDIT_SUM_OVERDUE").cast("double"),
    F.col("AMT_CREDIT_MAX_OVERDUE").cast("double"),
    F.col("AMT_CREDIT_SUM").cast("double")
)

print("Data selected successfully!")

Data selected successfully!


In [24]:
clean_app.printSchema()
clean_prev.printSchema()
clean_inst.printSchema()
clean_bureau.printSchema()

clean_app.show(5)
clean_prev.show(5)
clean_inst.show(5)
clean_bureau.show(5)

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- OCCUPATION_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)

root
 |-- SK_ID_PREV: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- NAME_CONTRACT_STATUS: string (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)

root
 |-- SK_ID_PREV: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- DAYS_INSTALMENT: do

+----------+----------+--------------------+----------+
|SK_ID_PREV|SK_ID_CURR|NAME_CONTRACT_STATUS|AMT_CREDIT|
+----------+----------+--------------------+----------+
|   1000149|    339127|            Approved|  231786.0|
|   1000190|    238250|            Approved| 1350000.0|
|   1000636|    135241|            Approved|   66897.0|
|   1001129|    140635|            Approved|   97902.0|
|   1001139|    166312|            Approved|   75568.5|
+----------+----------+--------------------+----------+
only showing top 5 rows



+----------+----------+---------------+------------------+--------------+-----------+
|SK_ID_PREV|SK_ID_CURR|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|
+----------+----------+---------------+------------------+--------------+-----------+
|   2164190|    100012|         -472.0|            -467.0|      11057.54|   11057.54|
|   2038692|    100013|        -1353.0|           -1353.0|        274.32|     274.32|
|   1251047|    100016|         -657.0|            -667.0|      14480.46|    14476.5|
|   1209367|    100049|          -29.0|             -29.0|        106.83|     106.83|
|   1950847|    100061|        -1332.0|           -1332.0|       6006.29|    6006.29|
+----------+----------+---------------+------------------+--------------+-----------+
only showing top 5 rows



+----------+-------------+-------------------+----------------------+----------------------+--------------+
|SK_ID_CURR|CREDIT_ACTIVE|AMT_CREDIT_SUM_DEBT|AMT_CREDIT_SUM_OVERDUE|AMT_CREDIT_MAX_OVERDUE|AMT_CREDIT_SUM|
+----------+-------------+-------------------+----------------------+----------------------+--------------+
|    328614|       Closed|                0.0|                   0.0|              20218.59|      234922.5|
|    290425|       Closed|                0.0|                   0.0|                   0.0|       70708.5|
|    437235|       Active|                0.0|                   0.0|                   0.0|       94557.6|
|    370199|       Closed|                0.0|                   0.0|                   0.0|      252000.0|
|    335518|       Closed|                0.0|                   0.0|                   0.0|       23355.0|
+----------+-------------+-------------------+----------------------+----------------------+--------------+
only showing top 5 rows

